# E/22/194

# Assignment 7c: Item Response Prediction and Click Through Rate Prediction

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

## 1. Visualising the 2PL response curves


$$p_i(\theta) = P(Y_i=1\mid\Theta=\theta) = \frac{1}{1+\exp[-a_i(\theta-b_i)]}$$

$$(a,b)=(0.7,0)$$

and

$$(a,b)=(2,-1),\ (2,0),\ (2,1)$$

In [3]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4, 4, 1000)

item_settings = [
    (0.7, 0.0, "a = 0.7, b = 0"),
    (2.0, -1.0, "a = 2.0, b = -1"),
    (2.0, 0.0, "a = 2.0, b = 0"),
    (2.0, 1.0, "a = 2.0, b = 1")
]

fig = go.Figure()

for a, b, label in item_settings:

    probability = 1 / (
        1 + np.exp(-a * (theta - b))
    )

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=probability,
            mode="lines",
            name=label
        )
    )

fig.add_hline(
    y=0.5,
    line_dash="dash",
    annotation_text="P(correct) = 0.5"
)

fig.update_layout(
    title="2PL Item Characteristic Curves",
    xaxis_title="Latent ability, θ",
    yaxis_title="P(Yᵢ = 1 | Θ = θ)",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

### Effect of $b_i$
At $\theta=b_i$,

$$p_i(b_i) = \frac{1}{1+e^0} = \frac12$$

Therefore,
$$ b_i=\text{ability level producing a }50\%\text{ success probability}$$

Also,
$$p(\theta;a,b+\Delta) = p(\theta-\Delta;a,b)$$

Hence,
$$ b_i\uparrow \quad\Longrightarrow\quad \text{curve shifts right}$$

A larger $b_i$ represents a more difficult item

### Effect of $a_i$
$$\frac{dp_i(\theta)}{d\theta} = a_i p_i(\theta)\bigl[1-p_i(\theta)\bigr]$$

At $\theta=b_i$,
$$p_i'(b_i)=\frac{a_i}{4}$$

Thus,
$$a_i\uparrow \quad\Longrightarrow\quad \text{steeper curve}$$




## 2. Sequential likelihood contribution

For one response $y_k\in\{0,1\}$,

$$L(y_k\mid\theta) = p_k(\theta)^{y_k} \left[1-p_k(\theta)\right]^{1-y_k}$$

For a correct answer,
$$y_k=1 \quad\Longrightarrow\quad L(1\mid\theta)=p_k(\theta)$$

For an incorrect answer,
$$y_k=0 \quad\Longrightarrow\quad L(0\mid\theta)=1-p_k(\theta)$$

Assuming conditional independence,
$$ L\left(\mathbf y^{(k)}\mid\theta\right) = \prod_{i=1}^{k} p_i(\theta)^{y_i} \left[1-p_i(\theta)\right]^{1-y_i}$$

The corresponding log-likelihood is:
$$ \log L\left(\mathbf y^{(k)}\mid\theta\right) = \sum_{i=1}^{k} \left[ y_i\log p_i(\theta) + (1-y_i)\log(1-p_i(\theta)) \right]$$

## 3. Recursive posterior update

Let
$$f_{k-1}(\theta) = f_{\Theta\mid\mathbf Y^{(k-1)}} \left( \theta\mid\mathbf y^{(k-1)} \right)$$

After observing $y_k$,
$$f_k(\theta) \propto L(y_k\mid\theta)f_{k-1}(\theta)$$

Therefore,
$$f_k(\theta) \propto p_k(\theta)^{y_k} [1-p_k(\theta)]^{1-y_k} f_{k-1}(\theta)$$

The normalised form is:
$$f_k(\theta) = \frac{L(y_k\mid\theta)f_{k-1}(\theta)}{\displaystyle \int_{\mathbb R} L(y_k\mid s)f_{k-1}(s)\,ds}$$

Initial state:
$$f_0(\theta) = \frac{1}{\sqrt{2\pi}} e^{-\theta^2/2}$$

Unlike Beta Binomial conjugacy, the Normal prior and logistic likelihood do not produce a standard closed form posterior. Numerical integration is required.



## 4. Correct answer to a difficult item

For $y_k=1$, the update is:
$$f_k(\theta) \propto p_k(\theta)f_{k-1}(\theta)$$

Since $\frac{dp_k(\theta)}{d\theta}>0$, larger values of $\theta$ receive greater weight.

For $\theta_H>\theta_L$,
$$\frac{f_k(\theta_H)}{f_k(\theta_L)} = \frac{p_k(\theta_H)}{p_k(\theta_L)} \frac{f_{k-1}(\theta_H)}{f_{k-1}(\theta_L)}$$

Because $\frac{p_k(\theta_H)}{p_k(\theta_L)}>1$, the posterior odds favour $\theta_H$.

For a difficult item, $b_k\gg0$, low ability values satisfy:
$$\theta\ll b_k \quad\Longrightarrow\quad p_k(\theta)\approx0$$

Therefore,
$$y_k=1,\ b_k\text{ large} \quad\Longrightarrow\quad \text{posterior shifts towards larger }\theta.$$

Low-ability regions are strongly reduced.

## 5. Discrimination and posterior sharpness

The item information is:

$$I_k(\theta) = a_k^2 p_k(\theta) \left[1-p_k(\theta)\right]$$

At $\theta=b_k$, $p_k(b_k)=\frac12$, so:

$$I_k(b_k)=\frac{a_k^2}{4}$$

Large discrimination($a_k\gg1$) gives $I_k(b_k)\gg0$. Hence,

$$\text{posterior variance decreases strongly near }\theta=b_k.$$

The posterior becomes narrower and sharper.

Small discrimination ($a_k\approx0$) gives $p_k(\theta)\approx\frac12$ over a wide range of $\theta$. Therefore,

$$L(y_k\mid\theta) \approx\text{constant}$$
Hence,
$$f_k(\theta)\approx f_{k-1}(\theta)$$

The response gives little information.

### Important condition
Even when $a_k$ is large,
$$\theta\text{ far from }b_k \quad\Longrightarrow\quad p_k(\theta)[1-p_k(\theta)]\approx0$$

Thus, the largest information occurs near:
$$\theta=b_k$$

## 6. Running grid approximation

Choose $\theta_1,\theta_2,\ldots,\theta_m$ on a fixed interval such as $[-4,4]$.
Let $\Delta\theta=\theta_{j+1}-\theta_j$.

**Initial grid density:**
$$q_j^{(0)} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right)$$

Normalise:
$$\widehat f_j^{(0)} = \frac{q_j^{(0)}}{\displaystyle \sum_{r=1}^{m} q_r^{(0)}\Delta\theta}$$

**Step $k$:**
Calculate:
$$p_{kj} = \frac{1}{1+\exp[-a_k(\theta_j-b_k)]}$$

Single-response likelihood:
$$\ell_{kj} = p_{kj}^{y_k} (1-p_{kj})^{1-y_k}$$

Unnormalised update:
$$q_j^{(k)} = \ell_{kj}\widehat f_j^{(k-1)}$$

Normalising constant:
$$c_k \approx \sum_{j=1}^{m} q_j^{(k)}\Delta\theta$$

Running posterior:
$$\widehat f_j^{(k)} = \frac{q_j^{(k)}}{c_k}$$

Posterior mean:
$$\widehat\theta_{\mathrm{Bayes}}^{(k)} \approx \sum_{j=1}^{m} \theta_j \widehat f_j^{(k)} \Delta\theta$$

MAP estimate:
$$\widehat\theta_{\mathrm{MAP}}^{(k)} = \theta_{\arg\max_j\widehat f_j^{(k)}}$$

Posterior variance:
$$ V_k \approx \sum_{j=1}^{m} \left( \theta_j- \widehat\theta_{\mathrm{Bayes}}^{(k)} \right)^2 \widehat f_j^{(k)} \Delta\theta$$

For numerical stability, use log values:

$$\log q_j^{(k)} = \log \widehat f_j^{(k-1)} + y_k\log p_{kj} + (1-y_k)\log(1-p_{kj})$$

Then subtract $\max_j\log q_j^{(k)}$ before exponentiation. This prevents underflow.

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

theta_true = 0.75
n_items = 20

theta_grid = np.linspace(-4.0, 4.0, 4001)

# ---------------------------------------------------------
# Generate item parameters
# ---------------------------------------------------------
b_values = rng.normal(
    loc=0.0,
    scale=1.0,
    size=n_items
)

a_values = rng.uniform(
    low=0.5,
    high=2.0,
    size=n_items
)

# ---------------------------------------------------------
# True probabilities and simulated responses
# ---------------------------------------------------------
true_probabilities = 1 / (
    1 + np.exp(
        -a_values * (theta_true - b_values)
    )
)

uniform_draws = rng.uniform(
    low=0.0,
    high=1.0,
    size=n_items
)

responses = (
    uniform_draws < true_probabilities
).astype(int)

# ---------------------------------------------------------
# Standard Normal prior
# ---------------------------------------------------------
log_posterior = (
    -0.5 * theta_grid**2
    -0.5 * np.log(2 * np.pi)
)

prior_density = np.exp(log_posterior)

prior_density /= np.trapezoid(
    prior_density,
    theta_grid
)

# Step 0 estimators
steps = [0]
posterior_means = [0.0]
map_estimates = [0.0]
posterior_variances = [1.0]

records = []

# ---------------------------------------------------------
# Sequential Bayesian updates
# ---------------------------------------------------------
for k in range(n_items):

    a_k = a_values[k]
    b_k = b_values[k]
    y_k = responses[k]

    # Response probability over the ability grid
    p_grid = 1 / (
        1 + np.exp(
            -a_k * (theta_grid - b_k)
        )
    )

    # Avoid log(0)
    p_grid = np.clip(
        p_grid,
        1e-12,
        1 - 1e-12
    )

    # Log-likelihood contribution
    log_likelihood = (
        y_k * np.log(p_grid)
        +
        (1 - y_k) * np.log1p(-p_grid)
    )

    # Running log-posterior
    log_posterior += log_likelihood

    # Stable exponentiation
    log_posterior -= np.max(log_posterior)

    posterior_density = np.exp(log_posterior)

    # Sequential normalization
    normalization_constant = np.trapezoid(
        posterior_density,
        theta_grid
    )

    posterior_density /= normalization_constant

    # Posterior mean
    posterior_mean = np.trapezoid(
        theta_grid * posterior_density,
        theta_grid
    )

    # MAP
    posterior_map = theta_grid[
        np.argmax(posterior_density)
    ]

    # Posterior variance
    posterior_variance = np.trapezoid(
        (
            theta_grid - posterior_mean
        )**2 * posterior_density,
        theta_grid
    )

    steps.append(k + 1)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )

    records.append({
        "Step": k + 1,
        "a_k": a_k,
        "b_k": b_k,
        "P_correct_true": true_probabilities[k],
        "Response": y_k,
        "Posterior_mean": posterior_mean,
        "MAP": posterior_map,
        "Posterior_variance": posterior_variance,
        "Mean_absolute_error":
            abs(posterior_mean - theta_true),
        "MAP_absolute_error":
            abs(posterior_map - theta_true)
    })

# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------
results = pd.DataFrame(records)

display(results.round(4))

print(
    "Number of correct responses:",
    int(np.sum(responses))
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior variance:",
    round(posterior_variances[-1], 4)
)

print(
    "Final posterior mean error:",
    round(
        abs(
            posterior_means[-1] - theta_true
        ),
        4
    )
)

print(
    "Final MAP error:",
    round(
        abs(
            map_estimates[-1] - theta_true
        ),
        4
    )
)

# ---------------------------------------------------------
# Estimator progression plot
# ---------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability = 0.75",
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Estimation of User Ability",
    xaxis_title="Number of answered items, k",
    yaxis_title="Ability estimate",
    template="plotly_white",
    width=950,
    height=550
)

fig.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig.show()

,Step,a_k,b_k,P_correct_true,Response,Posterior_mean,MAP,Posterior_variance,Mean_absolute_error,MAP_absolute_error
0,1,1.6371,0.3047,0.6746,1,0.6499,0.616,0.6777,0.1001,0.134
1,2,1.0318,-1.0400,0.8638,1,0.7609,0.706,0.6264,0.0109,0.044
2,3,1.9560,0.7505,0.4998,0,0.3362,0.358,0.4222,0.4138,0.392
3,4,1.8397,0.9406,0.4132,1,0.7675,0.758,0.3261,0.0175,0.008
4,5,1.6676,-1.9510,0.9891,1,0.7762,0.762,0.3213,0.0262,0.012
5,6,0.7920,-1.3022,0.8355,1,0.8181,0.798,0.3137,0.0681,0.048
6,7,1.2001,0.1278,0.6784,1,0.9272,0.894,0.2927,0.1772,0.144
7,8,0.5657,-0.3162,0.6464,1,0.9811,0.944,0.2904,0.2311,0.194
8,9,0.7314,-0.0168,0.6367,0,0.8450,0.822,0.2714,0.0950,0.072
9,10,1.5246,-0.8530,0.9201,1,0.8785,0.848,0.2610,0.1285,0.098


Number of correct responses: 14
Final posterior mean: 0.9208
Final MAP estimate: 0.902
Final posterior variance: 0.1431
Final posterior mean error: 0.1708
Final MAP error: 0.152


## Convergence analysis

Define the absolute errors:
$$e_{\mathrm{Bayes}}^{(k)} = \left| \widehat\theta_{\mathrm{Bayes}}^{(k)} - \theta_{\mathrm{true}} \right|$$
$$e_{\mathrm{MAP}}^{(k)} = \left| \widehat\theta_{\mathrm{MAP}}^{(k)} - \theta_{\mathrm{true}} \right|$$

The errors are not required to decrease at every step:
$$e^{(k+1)} \not\leq e^{(k)} \quad\text{for every }k.$$
A surprising response may temporarily move the estimate away from $\theta_{\mathrm{true}}$.

With informative items and increasing $k$,
$$\widehat\theta_{\mathrm{Bayes}}^{(k)} \rightarrow \theta_{\mathrm{true}}$$
$$\widehat\theta_{\mathrm{MAP}}^{(k)} \rightarrow \theta_{\mathrm{true}}$$
and generally,
$$\operatorname{Var} \left( \Theta\mid\mathbf Y^{(k)} \right) \downarrow$$

Therefore,

$$ k\uparrow \quad\Longrightarrow\quad \text{more accumulated evidence} \quad\Longrightarrow\quad \text{narrower posterior}.$$

For $n=20$, noticeable sampling variation may remain. A narrow posterior indicates greater confidence, but it does not guarantee that the estimate equals the true value exactly.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

## 1. Beta Distribution and Its Properties

The Beta density is
$$ f(\theta\mid\alpha,\beta) = \frac{1}{B(\alpha,\beta)} \theta^{\alpha-1}(1-\theta)^{\beta-1}, \qquad 0\leq\theta\leq1 $$
where
$$ B(\alpha,\beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)}. $$

Mean:
$$ \mathbb E[\Theta] = \frac{\alpha}{\alpha+\beta} $$

Variance:
$$ \operatorname{Var}(\Theta) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)} $$

Mode, when $\alpha>1, \beta>1$:
$$ \operatorname{Mode}(\Theta) = \frac{\alpha-1}{\alpha+\beta-2} $$

For the given distributions:
$$ \operatorname{Beta}(1,1): \qquad \mathbb E[\Theta]=0.5 $$
$$ \operatorname{Beta}(2,8): \qquad \mathbb E[\Theta]=0.2 $$
$$ \operatorname{Beta}(8,2): \qquad \mathbb E[\Theta]=0.8 $$

Hence,
$$
\begin{aligned}
\alpha=\beta &\Rightarrow \text{symmetric density} \\
\alpha<\beta &\Rightarrow \text{mass near } 0 \\
\alpha>\beta &\Rightarrow \text{mass near } 1
\end{aligned}
$$

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0.001, 0.999, 1000)

parameter_sets = [
    (1, 1, "Beta(1,1): Uniform"),
    (2, 8, "Beta(2,8): Right-skewed"),
    (8, 2, "Beta(8,2): Left-skewed")
]

fig = go.Figure()

for alpha, beta_param, label in parameter_sets:
    density = beta.pdf(
        theta,
        a=alpha,
        b=beta_param
    )

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=density,
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Beta Probability Density Functions",
    xaxis_title="Click-through rate, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

## 2. Sequential Likelihood and Joint History

For one response,
$$ L(y_k\mid\theta) = P(Y_k=y_k\mid\Theta=\theta) = \theta^{y_k}(1-\theta)^{1-y_k} $$
because $y_k\in\{0,1\}$.

Therefore,
$$ L(1\mid\theta)=\theta $$
and
$$ L(0\mid\theta)=1-\theta. $$

Assuming conditional independence,
$$ L(\mathbf y^{(k)}\mid\theta) = \prod_{i=1}^{k} \theta^{y_i}(1-\theta)^{1-y_i} = \theta^{\sum_{i=1}^{k}y_i} (1-\theta)^{\sum_{i=1}^{k}(1-y_i)}. $$

Using $S_k=\sum_{i=1}^{k}y_i$, we obtain
$$ L(\mathbf y^{(k)}\mid\theta) = \theta^{S_k}(1-\theta)^{k-S_k}. $$

Log-likelihood:
$$ \log L(\mathbf y^{(k)}\mid\theta) = S_k\log\theta + (k-S_k)\log(1-\theta). $$

## 3. Closed-Form Sequential Update

Assume that after $k-1$ impressions,
$$ \Theta\mid\mathbf y^{(k-1)} \sim \operatorname{Beta}(\alpha_{k-1},\beta_{k-1}). $$

Thus,
$$ f_{k-1}(\theta) = \frac{\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}}{B(\alpha_{k-1},\beta_{k-1})}. $$

Bayes’ theorem gives
$$ f_k(\theta) = \frac{L(y_k\mid\theta)f_{k-1}(\theta)}{\displaystyle \int_0^1 L(y_k\mid s)f_{k-1}(s)\,ds}. $$

Ignoring the normalising constant,
$$ f_k(\theta) \propto L(y_k\mid\theta)f_{k-1}(\theta). $$

Substitute:
$$ f_k(\theta) \propto \theta^{y_k}(1-\theta)^{1-y_k} \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1}. $$

Combine powers:
$$ f_k(\theta) \propto \theta^{\alpha_{k-1}+y_k-1} (1-\theta)^{\beta_{k-1}+1-y_k-1}. $$

Define
$$ \alpha_k=\alpha_{k-1}+y_k $$
and
$$ \beta_k=\beta_{k-1}+1-y_k. $$

Therefore,
$$ \Theta\mid\mathbf y^{(k)} \sim \operatorname{Beta}(\alpha_k,\beta_k). $$

The normalised posterior is
$$ f_k(\theta) = \frac{\theta^{\alpha_k-1}(1-\theta)^{\beta_k-1}}{B(\alpha_k,\beta_k)}. $$

After $k$ observations,
$$ \alpha_k = \alpha_0+\sum_{i=1}^{k}y_i = \alpha_0+S_k $$
$$ \beta_k = \beta_0+\sum_{i=1}^{k}(1-y_i) = \beta_0+k-S_k. $$

Thus,
$$ \operatorname{Beta\ prior} + \operatorname{Bernoulli\ data} \longrightarrow \operatorname{Beta\ posterior}. $$
This is Beta–Bernoulli conjugacy. When responses are grouped as a click count, it is commonly called Beta–Binomial conjugacy.

Posterior mean:
$$ \mathbb E \left[ \Theta\m

## 4. Dynamic Shifting Mechanics

### Case 1: Click
When $y_k=1$,
$$ \alpha_k=\alpha_{k-1}+1, \qquad \beta_k=\beta_{k-1}. $$

The posterior becomes
$$ f_k(\theta) \propto \theta f_{k-1}(\theta). $$

Since $\theta$ is larger near $1$,
$$ y_k=1 \Rightarrow \text{more posterior weight near larger } \theta. $$

For $\alpha_{k-1}>1, \beta_{k-1}>1$,
$$ \operatorname{Mode}_{k-1} = \frac{\alpha_{k-1}-1}{\alpha_{k-1}+\beta_{k-1}-2}. $$

After a click,
$$ \operatorname{Mode}_k = \frac{\alpha_{k-1}}{\alpha_{k-1}+\beta_{k-1}-1}. $$
The mode shifts towards $1$.

### Case 2: Non-click
When $y_k=0$,
$$ \alpha_k=\alpha_{k-1}, \qquad \beta_k=\beta_{k-1}+1. $$

The posterior becomes
$$ f_k(\theta) \propto (1-\theta)f_{k-1}(\theta). $$

Since $(1-\theta)$ is larger near $0$,
$$ y_k=0 \Rightarrow \text{more posterior weight near smaller } \theta. $$

After a non-click,
$$ \operatorname{Mode}_k = \frac{\alpha_{k-1}-1}{\alpha_{k-1}+\beta_{k-1}-1}. $$
The mode shifts towards $0$.

### Comparison with the 2PL model
**Beta–Bernoulli model:**
$$ (\alpha_{k-1},\beta_{k-1}) \longrightarrow (\alpha_k,\beta_k) $$
using simple arithmetic. No numerical integration is required.

**2PL IRT model:**
$$ f_k(\theta) \propto p_k(\theta)^{y_k} [1-p_k(\theta)]^{1-y_k} f_{k-1}(\theta). $$

The logistic likelihood multiplied by a Normal prior does not produce a standard posterior family.

Therefore,
$$ \text{2PL model} \Rightarrow \text{grid integration, MCMC, or variational inference}. $$

## 5. Running Point Estimators

Posterior mean:
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k} $$
or
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_0+S_k}{\alpha_0+\beta_0+k}. $$

MAP estimate:
When $\alpha_k>1, \beta_k>1$,
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2}. $$

Boundary cases:
$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)} =
\begin{cases}
\dfrac{\alpha_k-1}{\alpha_k+\beta_k-2}, &\alpha_k>1, \beta_k>1, \\[10pt]
0, &\alpha_k\leq1, \beta_k>1, \\[4pt]
1, &\alpha_k>1, \beta_k\leq1.
\end{cases}
$$

For $\alpha_k=\beta_k=1$, the distribution is uniform:
$$ f(\theta)=1. $$
Every value in $[0,1]$ is a mode. For the initial plotted value, use
$$ \widehat{\theta}_{\mathrm{MAP}}^{(0)}=0.5. $$

Posterior variance:
$$ V_k = \operatorname{Var}\left(\Theta\mid\mathbf y^{(k)}\right) = \frac{\alpha_k\beta_k}{(\alpha_k+\beta_k)^2(\alpha_k+\beta_k+1)}. $$

## 6. Sequential CTR Simulation

Given
$$ \theta_{\mathrm{true}}=0.35, \qquad n=100, \qquad \alpha_0=1, \qquad \beta_0=1. $$

Generate
$$ U_k\sim\operatorname{Uniform}(0,1). $$

Then
$$
y_k=
\begin{cases}
1, & U_k<0.35 \\
0, & U_k\geq0.35.
\end{cases}

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

theta_true = 0.35
n_impressions = 100

alpha_0 = 1.0
beta_0 = 1.0


# ---------------------------------------------------------
# Beta MAP function
# ---------------------------------------------------------
def beta_map(alpha, beta_param):
    if alpha > 1 and beta_param > 1:
        return (
            (alpha - 1)
            /
            (alpha + beta_param - 2)
        )

    if alpha == 1 and beta_param == 1:
        # Uniform distribution:
        # every value is a mode.
        # Use 0.5 only for plotting.
        return 0.5

    if alpha <= 1 and beta_param > 1:
        return 0.0

    if alpha > 1 and beta_param <= 1:
        return 1.0

    return np.nan


# ---------------------------------------------------------
# Initial state: k = 0
# ---------------------------------------------------------
alpha_k = alpha_0
beta_k = beta_0

steps = [0]

posterior_means = [
    alpha_k / (alpha_k + beta_k)
]

map_estimates = [
    beta_map(alpha_k, beta_k)
]

posterior_variances = [
    (
        alpha_k * beta_k
        /
        (
            (alpha_k + beta_k)**2
            * (alpha_k + beta_k + 1)
        )
    )
]

records = []


# ---------------------------------------------------------
# Sequential updates
# ---------------------------------------------------------
for k in range(1, n_impressions + 1):

    # Uniform random draw
    u_k = rng.uniform(0.0, 1.0)

    # Simulated click
    y_k = int(u_k < theta_true)

    # Beta posterior updates
    alpha_k = alpha_k + y_k
    beta_k = beta_k + 1 - y_k

    # Posterior mean
    posterior_mean = (
        alpha_k
        /
        (alpha_k + beta_k)
    )

    # MAP estimate
    posterior_map = beta_map(
        alpha_k,
        beta_k
    )

    # Posterior variance
    posterior_variance = (
        alpha_k * beta_k
        /
        (
            (alpha_k + beta_k)**2
            * (alpha_k + beta_k + 1)
        )
    )

    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )

    records.append({
        "Step": k,
        "Uniform draw": u_k,
        "Response": y_k,
        "Alpha": alpha_k,
        "Beta": beta_k,
        "Posterior mean": posterior_mean,
        "MAP": posterior_map,
        "Posterior variance": posterior_variance,
        "Mean absolute error":
            abs(posterior_mean - theta_true),
        "MAP absolute error":
            abs(posterior_map - theta_true)
    })


# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------
results = pd.DataFrame(records)

display(results.round(4))

total_clicks = int(
    alpha_k - alpha_0
)

total_non_clicks = int(
    beta_k - beta_0
)

print("Total clicks:", total_clicks)
print("Total non-clicks:", total_non_clicks)

print(
    "Final posterior:",
    f"Beta({alpha_k:.0f}, {beta_k:.0f})"
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior variance:",
    round(posterior_variances[-1], 6)
)


# ---------------------------------------------------------
# Plot estimator progression
# ---------------------------------------------------------
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35",
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Bayesian Tracking of Advertisement CTR",
    xaxis_title="Number of impressions, k",
    yaxis_title="Estimated CTR",
    template="plotly_white",
    width=1000,
    height=560
)

fig.update_xaxes(
    tickmode="linear",
    dtick=10
)

fig.update_yaxes(
    range=[0, 1]
)

fig.show()

,Step,Uniform draw,Response,Alpha,Beta,Posterior mean,MAP,Posterior variance,Mean absolute error,MAP absolute error
0,1,0.7740,0,1.0,2.0,0.3333,0.0000,0.0556,0.0167,0.3500
1,2,0.4389,0,1.0,3.0,0.2500,0.0000,0.0375,0.1000,0.3500
2,3,0.8586,0,1.0,4.0,0.2000,0.0000,0.0267,0.1500,0.3500
3,4,0.6974,0,1.0,5.0,0.1667,0.0000,0.0198,0.1833,0.3500
4,5,0.0942,1,2.0,5.0,0.2857,0.2000,0.0255,0.0643,0.1500
...,...,...,...,...,...,...,...,...,...,...
95,96,0.6303,0,32.0,66.0,0.3265,0.3229,0.0022,0.0235,0.0271
96,97,0.3618,0,32.0,67.0,0.3232,0.3196,0.0022,0.0268,0.0304
97,98,0.0876,1,33.0,67.0,0.3300,0.3265,0.0022,0.0200,0.0235
98,99,0.1180,1,34.0,67.0,0.3366,0.3333,0.0022,0.0134,0.0167


Total clicks: 33
Total non-clicks: 67
Final posterior: Beta(34, 68)
Final posterior mean: 0.3333
Final MAP estimate: 0.33
Final posterior variance: 0.002157


## 7. Numerical Result for Seed (42)

The simulation gives $S_{100}=33$ clicks and $100-33=67$ non-clicks.
Thus,
$$ \alpha_{100} = 1+33 = 34 $$
$$ \beta_{100} = 1+67 = 68 $$

The final posterior is
$$ \Theta\mid\mathbf y^{(100)} \sim \operatorname{Beta}(34,68). $$

Posterior mean:
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(100)} = \frac{34}{34+68} = 0.3333. $$

MAP:
$$ \widehat{\theta}_{\mathrm{MAP}}^{(100)} = \frac{34-1}{34+68-2} = \frac{33}{100} = 0.3300. $$

Posterior variance:
$$ V_{100} = \frac{(34)(68)}{(102)^2(103)} \approx 0.002157. $$

Posterior standard deviation:
$$ \sqrt{V_{100}} \approx 0.0464. $$



Therefore,
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{S_k}{k} \rightarrow 0.35. $$

The estimation error may increase temporarily:
$$ e_{k+1} \not\leq e_k \quad\text{for every individual step}. $$
Random clicks and non-clicks cause fluctuations.

Generally,
$$
\begin{aligned}
k\uparrow &\Rightarrow \text{prior influence}\downarrow \\
k\uparrow &\Rightarrow \text{data influence}\uparrow \\
k\uparrow &\Rightarrow \text{posterior variance}\downarrow \\
k\uparrow &\Rightarrow \text{confidence in CTR}\uparrow
\end{aligned}
$$

Therefore,
$$ \text{accumulated evidence gradually dominates the initial prior}. $$

## 8. Convergence Analysis

For a general prior,
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_0+S_k}{\alpha_0+\beta_0+k}. $$

Let
$$ m_0 = \frac{\alpha_0}{\alpha_0+\beta_0} $$
be the prior mean and
$$ \overline y_k = \frac{S_k}{k} $$
be the observed click rate.

Then
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_0+\beta_0}{\alpha_0+\beta_0+k} m_0 + \frac{k}{\alpha_0+\beta_0+k} \overline y_k. $$

For $\alpha_0=\beta_0=1$, we have $m_0=0.5$. Hence,
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{2}{k+2}(0.5) + \frac{k}{k+2} \left(\frac{S_k}{k}\right). $$

Prior weight:
$$ w_{\mathrm{prior}}(k) = \frac{2}{k+2}. $$

Data weight:
$$ w_{\mathrm{data}}(k) = \frac{k}{k+2}. $$

As $k\rightarrow\infty$,
$$ \frac{2}{k+2}\rightarrow0, \qquad \frac{k}{k+2}\rightarrow1. $$

Therefore,
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} \rightarrow \frac{S_k}{k}. $$

By the law of large numbers, $\frac{S_k}{k} \rightarrow \theta_{\mathrm{true}}$. Thus,
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} \rightarrow 0.35. $$

For the uniform prior, after observing at least one click and one non-click,
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{(1+S_k)-1}{(1+S_k)+(1+k-S_k)-2} = \frac{S_k}{k}. $$

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

## 1. Initial Prior

$$ \Theta\sim\operatorname{Beta}(8,1.5). $$

The density is
$$ f_\Theta^{(0)}(\theta) = \frac{1}{B(8,1.5)} \theta^{8-1}(1-\theta)^{1.5-1}, \qquad 0<\theta<1. $$

### Prior mean
For
$$ \Theta\sim\operatorname{Beta}(\alpha,\beta), $$
$$ \mathbb E[\Theta] = \frac{\alpha}{\alpha+\beta}. $$

Therefore,
$$ \mathbb E[\Theta^{(0)}] = \frac{8}{8+1.5} = \frac{8}{9.5}. $$
$$ \mathbb E[\Theta^{(0)}] \approx 0.8421. $$

### Prior mode
Since
$$ \alpha>1, \qquad \beta>1, $$
$$ \operatorname{Mode}(\Theta) = \frac{\alpha-1}{\alpha+\beta-2}. $$

Thus,
$$ \operatorname{Mode}(\Theta) = \frac{7}{7.5}. $$
$$ \operatorname{Mode}(\Theta) \approx 0.9333. $$

Hence,
$$ \text{prior mass is concentrated near healthy stiffness values}. $$

In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 5000)

alpha_0 = 8.0
beta_0 = 1.5

prior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

prior_density /= np.trapezoid(
    prior_density,
    theta_grid
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5)"
    )
)

fig.add_vline(
    x=alpha_0 / (alpha_0 + beta_0),
    line_dash="dash",
    annotation_text="Prior mean = 0.8421"
)

fig.add_vline(
    x=(alpha_0 - 1) / (alpha_0 + beta_0 - 2),
    line_dash="dot",
    annotation_text="Prior mode = 0.9333"
)

fig.update_layout(
    title="Initial Prior for Remaining Stiffness",
    xaxis_title="Remaining stiffness factor, θ",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=520
)

fig.show()

## 2. Structural Likelihood

From
$$ \ln Y_k\mid\Theta=\theta \sim \mathscr N \left( \ln(\theta K_{\text{nominal}}), \sigma^2 \right), $$
the measurement follows a log-normal distribution:
$$ Y_k\mid\Theta=\theta \sim \operatorname{LogNormal} \left( \ln(\theta K_{\text{nominal}}), \sigma^2 \right). $$

Therefore,
$$ L(y_k\mid\theta) = \frac{1}{y_k\sigma\sqrt{2\pi}} \exp \left[ -\frac{ \left( \ln y_k-\ln(\theta K_{\text{nominal}}) \right)^2 }{ 2\sigma^2 } \right], \quad y_k>0. $$

Equivalent form:
$$ L(y_k\mid\theta) = \frac{1}{y_k\sigma\sqrt{2\pi}} \exp \left[ -\frac{ \left( \ln\frac{y_k}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]. $$

For conditionally independent measurements,
$$ L\left(\mathbf y^{(k)}\mid\theta\right) = \prod_{i=1}^{k} L(y_i\mid\theta). $$

Thus,
$$ L\left(\mathbf y^{(k)}\mid\theta\right) = \prod_{i=1}^{k} \frac{1}{y_i\sigma\sqrt{2\pi}} \exp \left[ -\frac{ \left( \ln\frac{y_i}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]. $$

Log-likelihood:
$$ \log L = -k\log(\sigma\sqrt{2\pi}) -\sum_{i=1}^{k}\log y_i -\frac{1}{2\sigma^2} \sum_{i=1}^{k} \left( \ln\frac{y_i}{\theta K_{\text{nominal}}} \right)^2. $$

## 3. Non-Conjugate Posterior Update

At step $k-1$,
$$ f_{k-1}(\theta) = f_{\Theta\mid\mathbf Y^{(k-1)}} \left( \theta\mid\mathbf y^{(k-1)} \right). $$

After observing $y_k$,
$$ f_k(\theta) \propto L(y_k\mid\theta)f_{k-1}(\theta). $$

The normalised posterior is
$$ f_k(\theta) = \frac{L(y_k\mid\theta)f_{k-1}(\theta)}{\displaystyle \int_0^1 L(y_k\mid s)f_{k-1}(s)\,ds}. $$

Substituting the likelihood,
$$ f_k(\theta) \propto f_{k-1}(\theta) \exp \left[ -\frac{ \left( \ln\frac{y_k}{\theta K_{\text{nominal}}} \right)^2 }{ 2\sigma^2 } \right]. $$

The factor
$$ \exp \left[ -\frac{(\ln\theta)^2+\cdots}{2\sigma^2} \right] $$
does not have the Beta kernel form
$$ \theta^{\alpha-1}(1-\theta)^{\beta-1}. $$

Therefore,
$$ \operatorname{Beta\ prior} + \operatorname{lognormal\ likelihood} \not\Rightarrow \operatorname{Beta\ posterior}. $$

Hence,
$$ \text{numerical integration is required}. $$

## 4. Running Point Estimates

Posterior mean:
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta\mid\mathbf y^{(k)}] = \int_0^1 \theta f_k(\theta)\,d\theta. $$

MAP estimate:
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname*{arg\,max}_{0<\theta\leq1} f_k(\theta). $$

Posterior variance:
$$ V_k = \int_0^1 \left( \theta - \widehat{\theta}_{\mathrm{Bayes}}^{(k)} \right)^2 f_k(\theta)\,d\theta. $$

Posterior standard deviation:
$$ s_k=\sqrt{V_k}. $$



## 5. Grid Approximation

Choose $M$ grid points:
$$ \theta_j = 0.01+ (j-1)\Delta\theta, \qquad j=1,\ldots,M, $$
where
$$ \Delta\theta = \frac{1-0.01}{M-1}. $$
Using $0.01$ avoids $\ln(0)$.

Initial density:
$$ q_j^{(0)} = \frac{1}{B(8,1.5)} \theta_j^7(1-\theta_j)^{0.5}. $$

Normalisation:
$$ Z_0 \approx \operatorname{trapz} \left( q^{(0)},\theta \right). $$
$$ f_j^{(0)} = \frac{q_j^{(0)}}{Z_0}. $$

Sequential update:
For measurement $y_k$,
$$ \ell_{kj} = L(y_k\mid\theta_j). $$

Unnormalised posterior:
$$ q_j^{(k)} = \ell_{kj}f_j^{(k-1)}. $$

Normalising constant:
$$ Z_k \approx \operatorname{trapz} \left( q^{(k)},\theta \right). $$

Normalised posterior:
$$ f_j^{(k)} = \frac{q_j^{(k)}}{Z_k}. $$

Grid posterior mean:
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(k)} \approx \operatorname{trapz} \left( \theta_jf_j^{(k)},\theta_j \right). $$

Grid MAP:
$$ \widehat{\theta}_{\mathrm{MAP}}^{(k)} = \theta_{\arg\max_j f_j^{(k)}}. $$

For numerical stability:
$$ \log q_j^{(k)} = \log f_j^{(k-1)} + \log L(y_k\mid\theta_j). $$

Let $m_k=\max_j\log q_j^{(k)}$. Then
$$ \widetilde q_j^{(k)} = \exp \left( \log q_j^{(k)}-m_k \right). $$

Finally,
$$ f_j^{(k)} = \frac{\widetilde q_j^{(k)}}{\operatorname{trapz}(\widetilde q^{(k)},\theta)}. $$



## 6. Complete Simulation

Given:
$$ \theta_{\text{true}}=0.68, $$
$$ K_{\text{nominal}}=50.0\ \text{kN/mm}, $$
$$ \sigma=0.15, $$
$$ n=15. $$

Generate:
$$ \epsilon_k\sim\mathscr N(0,0.15^2), $$
$$ y_k = 0.68(50)e^{\epsilon_k}. $$

In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import beta

# =========================================================
# Settings
# =========================================================
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_measurements = 15

alpha_0 = 8.0
beta_0 = 1.5

theta_grid = np.linspace(
    0.01,
    1.0,
    5000
)

milestone_steps = [0, 1, 2, 5, 10, 15]

# =========================================================
# Initial Beta prior
# =========================================================
posterior_density = beta.pdf(
    theta_grid,
    a=alpha_0,
    b=beta_0
)

posterior_density /= np.trapezoid(
    posterior_density,
    theta_grid
)

# =========================================================
# Simulate log-normal sensor readings
# =========================================================
epsilon_values = rng.normal(
    loc=0.0,
    scale=sigma,
    size=n_measurements
)

sensor_readings = (
    theta_true
    * K_nominal
    * np.exp(epsilon_values)
)

# =========================================================
# Initial estimators: step 0
# =========================================================
initial_mean = np.trapezoid(
    theta_grid * posterior_density,
    theta_grid
)

initial_map = theta_grid[
    np.argmax(posterior_density)
]

initial_variance = np.trapezoid(
    (
        theta_grid - initial_mean
    )**2
    * posterior_density,
    theta_grid
)

steps = [0]
posterior_means = [initial_mean]
map_estimates = [initial_map]
posterior_variances = [initial_variance]
posterior_std = [np.sqrt(initial_variance)]

milestone_densities = {
    0: posterior_density.copy()
}

records = []

# =========================================================
# Sequential bounded-grid updates
# =========================================================
for k, y_k in enumerate(
    sensor_readings,
    start=1
):

    # -----------------------------------------------------
    # Log-likelihood over the theta grid
    # -----------------------------------------------------
    log_likelihood = (
        -np.log(
            y_k * sigma * np.sqrt(2 * np.pi)
        )
        -
        (
            np.log(
                y_k
                /
                (
                    theta_grid * K_nominal
                )
            )**2
        )
        /
        (2 * sigma**2)
    )

    # -----------------------------------------------------
    # Stable log-posterior update
    # -----------------------------------------------------
    log_previous = np.log(
        posterior_density + 1e-300
    )

    log_unnormalized = (
        log_previous + log_likelihood
    )

    log_unnormalized -= np.max(
        log_unnormalized
    )

    unnormalized_density = np.exp(
        log_unnormalized
    )

    # -----------------------------------------------------
    # Trapezoidal normalization
    # -----------------------------------------------------
    normalization_constant = np.trapezoid(
        unnormalized_density,
        theta_grid
    )

    posterior_density = (
        unnormalized_density
        /
        normalization_constant
    )

    # -----------------------------------------------------
    # Posterior mean
    # -----------------------------------------------------
    posterior_mean = np.trapezoid(
        theta_grid * posterior_density,
        theta_grid
    )

    # -----------------------------------------------------
    # MAP estimate
    # -----------------------------------------------------
    posterior_map = theta_grid[
        np.argmax(posterior_density)
    ]

    # -----------------------------------------------------
    # Posterior variance
    # -----------------------------------------------------
    posterior_variance = np.trapezoid(
        (
            theta_grid - posterior_mean
        )**2
        * posterior_density,
        theta_grid
    )

    standard_deviation = np.sqrt(
        posterior_variance
    )

    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(posterior_map)
    posterior_variances.append(
        posterior_variance
    )
    posterior_std.append(
        standard_deviation
    )

    if k in milestone_steps:
        milestone_densities[k] = (
            posterior_density.copy()
        )

    records.append({
        "Step": k,
        "Sensor reading": y_k,
        "Posterior mean": posterior_mean,
        "MAP": posterior_map,
        "Posterior variance":
            posterior_variance,
        "Posterior SD":
            standard_deviation,
        "Mean error":
            abs(
                posterior_mean
                - theta_true
            ),
        "MAP error":
            abs(
                posterior_map
                - theta_true
            )
    })

# =========================================================
# Results table
# =========================================================
results = pd.DataFrame(records)

display(results.round(5))

print(
    "Initial prior mean:",
    round(initial_mean, 4)
)

print(
    "Initial prior MAP:",
    round(initial_map, 4)
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior variance:",
    round(posterior_variances[-1], 6)
)

print(
    "Final posterior SD:",
    round(posterior_std[-1], 4)
)

# =========================================================
# Confidence criterion
# =========================================================
confidence_step = None

for k in range(1, n_measurements + 1):

    mean_close = (
        abs(
            posterior_means[k]
            - theta_true
        )
        <= 0.02
    )

    map_close = (
        abs(
            map_estimates[k]
            - theta_true
        )
        <= 0.02
    )

    sufficiently_narrow = (
        posterior_std[k]
        <= 0.05
    )

    if (
        mean_close
        and map_close
        and sufficiently_narrow
    ):
        confidence_step = k
        break

print(
    "First confidence step:",
    confidence_step
)

# =========================================================
# Approximate final 95% credible interval
# =========================================================
cdf = np.concatenate(
    (
        [0.0],
        np.cumsum(
            (
                posterior_density[:-1]
                +
                posterior_density[1:]
            )
            / 2
            * np.diff(theta_grid)
        )
    )
)

cdf /= cdf[-1]

lower_95 = np.interp(
    0.025,
    cdf,
    theta_grid
)

upper_95 = np.interp(
    0.975,
    cdf,
    theta_grid
)

print(
    "Final 95% credible interval:",
    (
        round(lower_95, 4),
        round(upper_95, 4)
    )
)

# =========================================================
# Plot 1: posterior density milestones
# =========================================================
fig_density = go.Figure()

for step in milestone_steps:

    fig_density.add_trace(
        go.Scatter(
            x=theta_grid,
            y=milestone_densities[step],
            mode="lines",
            name=f"k = {step}"
        )
    )

fig_density.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68"
)

fig_density.update_layout(
    title=(
        "Sequential Posterior Density "
        "for Remaining Stiffness"
    ),
    xaxis_title="Remaining stiffness factor, θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    width=1000,
    height=600
)

fig_density.show()

# =========================================================
# Plot 2: estimator timeline
# =========================================================
fig_timeline = go.Figure()

fig_timeline.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig_timeline.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig_timeline.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68",
    annotation_position="top right"
)

fig_timeline.update_layout(
    title=(
        "Sequential Estimation of "
        "Remaining Structural Stiffness"
    ),
    xaxis_title="Number of sensor readings, k",
    yaxis_title="Estimated stiffness factor",
    template="plotly_white",
    width=1000,
    height=550
)

fig_timeline.update_xaxes(
    tickmode="linear",
    dtick=1
)

fig_timeline.update_yaxes(
    range=[0, 1]
)

fig_timeline.show()

,Step,Sensor reading,Posterior mean,MAP,Posterior variance,Posterior SD,Mean error,MAP error
0,1,35.59012,0.79468,0.79721,0.00858,0.09263,0.11468,0.11721
1,2,29.08908,0.69767,0.68769,0.00517,0.07189,0.01767,0.00769
2,3,38.05103,0.71774,0.71066,0.00371,0.06090,0.03774,0.03066
3,4,39.15175,0.73317,0.72770,0.00292,0.05407,0.05317,0.04770
4,5,25.37350,0.68229,0.67799,0.00206,0.04543,0.00229,0.00201
5,6,27.96723,0.66036,0.65680,0.00162,0.04024,0.01964,0.02320
6,7,34.65828,0.66491,0.66175,0.00141,0.03754,0.01509,0.01825
7,8,32.42482,0.66286,0.66016,0.00123,0.03503,0.01714,0.01984
8,9,33.91442,0.66455,0.66214,0.00110,0.03312,0.01545,0.01786
9,10,29.91631,0.65766,0.65541,0.00097,0.03111,0.02234,0.02459


Initial prior mean: 0.8421
Initial prior MAP: 0.9333
Final posterior mean: 0.6873
Final MAP estimate: 0.6857
Final posterior variance: 0.000705
Final posterior SD: 0.0266
First confidence step: 5
Final 95% credible interval: (np.float64(0.6367), np.float64(0.7408))


## 7. Numerical Results for Seed (42)

Approximate milestone results:

| $k$ | Posterior mean | MAP | Posterior SD |
|---|---|---|---|
| 0 | 0.8421 | 0.9333 | 0.1125 |
| 1 | 0.7947 | 0.7972 | 0.0926 |
| 2 | 0.6977 | 0.6877 | 0.0719 |
| 5 | 0.6823 | 0.6780 | 0.0454 |
| 10 | 0.6577 | 0.6554 | 0.0311 |
| 15 | 0.6873 | 0.6857 | 0.0266 |

Final posterior estimates:
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(15)} \approx 0.6873 $$
$$ \widehat{\theta}_{\mathrm{MAP}}^{(15)} \approx 0.6857 $$
$$ V_{15} \approx 0.000705 $$
$$ s_{15} \approx 0.0266. $$

Approximate 95% credible interval:
$$ \Theta\mid\mathbf y^{(15)} \in [0.6367,\ 0.7408] $$
with approximately 95% posterior probability.



## 8. Number of Readings Required

Confidence must be defined mathematically. Use the criterion:
$$ \left| \widehat{\theta}_{\mathrm{Bayes}}^{(k)} - 0.68 \right| \leq 0.02, $$
$$ \left| \widehat{\theta}_{\mathrm{MAP}}^{(k)} - 0.68 \right| \leq 0.02, $$
and
$$ s_k \leq 0.05. $$

For seed 42,
$$ k=5 $$
is the first step satisfying all three conditions.
At $k=5$,
$$ \widehat{\theta}_{\mathrm{Bayes}}^{(5)} \approx 0.6823, $$
$$ \widehat{\theta}_{\mathrm{MAP}}^{(5)} \approx 0.6780, $$
$$ s_5 \approx 0.0454. $$

Thus,
$$ \text{about five readings overcome the optimistic healthy prior in this simulation}. $$

This number depends on:
$$ \sigma, \qquad \theta_{\text{true}}, \qquad \text{prior strength}, \qquad \text{random sensor readings}. $$



## 9. Structural Safety Interpretation

Let $\theta_{\text{critical}}$ be a safety threshold. The posterior probability of unsafe stiffness is
$$ P( \Theta<\theta_{\text{critical}} \mid\mathbf y^{(k)} ) = \int_0^{\theta_{\text{critical}}} f_k(\theta)\,d\theta. $$

A broad posterior gives uncertain classification:
$$ P( \Theta<\theta_{\text{critical}} \mid\mathbf y^{(k)} ) \approx 0.5. $$

A narrow posterior below the threshold gives
$$ P( \Theta<\theta_{\text{critical}} \mid\mathbf y^{(k)} ) \rightarrow 1. $$

A narrow posterior above the threshold gives
$$ P( \Theta<\theta_{\text{critical}} \mid\mathbf y^{(k)} ) \rightarrow 0. $$

Therefore,
$$ \text{narrowing density} \Rightarrow \text{reduced uncertainty} \Rightarrow \text{more decisive maintenance action}. $$

For example, with $\theta_{\text{critical}}=0.75$, the final simulation gives approximately
$$ P(\Theta<0.75\mid\mathbf y^{(15)}) \approx 0.989. $$

This gives strong evidence that the remaining stiffness is below 75%.

# Q. Gaussian Mixture Clustering as Conditional Updating


## 1. Marginal Density

By the law of total probability,
$$ p(x_i) = \sum_{k=1}^{K} p(x_i,C_i=k). $$

Using
$$ p(x_i,C_i=k) = p(x_i\mid C_i=k)P(C_i=k), $$
$$ p(x_i) = \sum_{k=1}^{K} p(x_i\mid C_i=k)P(C_i=k). $$

Substitute
$$ p(x_i\mid C_i=k) = \mathscr N(x_i\mid\mu_k,\Sigma_k) $$
and
$$ P(C_i=k)=\phi_k. $$

Therefore,
$$ p(x_i) = \sum_{k=1}^{K} \phi_k \mathscr N(x_i\mid\mu_k,\Sigma_k) $$
subject to
$$ \phi_k\geq0, \qquad \sum_{k=1}^{K}\phi_k=1. $$

It is a Gaussian mixture because
$$ \text{overall density} = \sum_{k=1}^{K} \text{weight}_k \times \text{Gaussian density}_k. $$

A mixture can represent
$$ \text{multiple modes}, \qquad \text{different centres}, \qquad \text{different covariance shapes}. $$

## 2. Posterior Cluster Probability

For continuous $X_i$, use the conditional density $p(x_i\mid C_i=k)$. Bayes’ rule gives

$$ P(C_i=k\mid X_i=x_i) = \frac{ p(x_i\mid C_i=k)P(C_i=k) }{ p(x_i) } $$

Using
$$ p(x_i) = \sum_{j=1}^{K} p(x_i\mid C_i=j)P(C_i=j), $$
$$ P(C_i=k\mid X_i=x_i) = \frac{ p(x_i\mid C_i=k)P(C_i=k) }{ \displaystyle \sum_{j=1}^{K} p(x_i\mid C_i=j)P(C_i=j) } $$

Substitution gives

$$ \gamma_{ik} = P(C_i=k\mid X_i=x_i) = \frac{ \phi_k \mathscr N(x_i\mid\mu_k,\Sigma_k) }{ \displaystyle \sum_{j=1}^{K} \phi_j \mathscr N(x_i\mid\mu_j,\Sigma_j) } $$

The numerator is
$$ \text{prior cluster probability} \times \text{compatibility of }x_i\text{ with cluster }k. $$

The denominator is the evidence:
$$ p(x_i) = \sum_{j=1}^{K} \phi_j \mathscr N(x_i\mid\mu_j,\Sigma_j). $$

Properties:
$$ 0\leq\gamma_{ik}\leq1, $$
$$ \sum_{k=1}^{K}\gamma_{ik}=1 $$

Thus,
$$ \gamma_{ik} = \text{posterior probability that }x_i \text{ belongs to cluster }k. $$

## 3. One-Hot Latent Variable

Define
$$ Z_{ik} = \begin{cases} 1,&C_i=k,\\ 0,&C_i\neq k. \end{cases} $$

Since $Z_{ik}$ is an indicator variable,
$$ \begin{aligned} \mathbb E[Z_{ik}\mid X_i=x_i] &= 1\cdot P(Z_{ik}=1\mid X_i=x_i)\\ &\quad+ 0\cdot P(Z_{ik}=0\mid X_i=x_i). \end{aligned} $$

Therefore,
$$ \mathbb E[Z_{ik}\mid X_i=x_i] = P(Z_{ik}=1\mid X_i=x_i). $$

Since $Z_{ik}=1 \iff C_i=k$,
$$ \mathbb E[Z_{ik}\mid X_i=x_i] = P(C_i=k\mid X_i=x_i) = \gamma_{ik}. $$

For
$$ Z_i = \begin{bmatrix} Z_{i1}\\ Z_{i2}\\ \vdots\\ Z_{iK} \end{bmatrix}, $$
$$ \mathbb E[Z_i\mid X_i=x_i] = \begin{bmatrix} \gamma_{i1}\\ \gamma_{i2}\\ \vdots\\ \gamma_{iK} \end{bmatrix}. $$

Hence,
$$ \text{soft assignment of }x_i = \mathbb E[Z_i\mid X_i=x_i]. $$

Example:
$$ \mathbb E[Z_i\mid X_i=x_i] = \begin{bmatrix} 0.48\\ 0.49\\ 0.03 \end{bmatrix}. $$

This means
$$ P(C_i=1\mid x_i)=0.48, $$
$$ P(C_i=2\mid x_i)=0.49, $$
$$ P(C_i=3\mid x_i)=0.03. $$

## 4. Soft and Hard Clustering

**Soft clustering**
$$ \boldsymbol\gamma_i = (\gamma_{i1},\ldots,\gamma_{iK})^T. $$
All posterior membership probabilities are retained.

**Hard clustering**
$$ \widehat C_i = \operatorname*{arg\,max}_{1\leq k\leq K} \gamma_{ik}. $$

For
$$ \boldsymbol\gamma_i = (0.48,0.49,0.03)^T, $$
$$ \widehat C_i=2. $$

Soft clustering preserves $0.48\approx0.49$. Hard clustering preserves only $\widehat C_i=2$.
Therefore,
$$ \text{hard clustering discards posterior uncertainty}. $$

## 5. Conditional Expectation Given the Cluster

The model states
$$ X_i\mid C_i=k \sim \mathscr N(\mu_k,\Sigma_k). $$

The expectation of a multivariate Gaussian is its mean:
$$ \mathbb E[X_i\mid C_i=k]=\mu_k. $$

Therefore,
$$ \mu_k=\text{centre of cluster }k. $$

Comparison:
$$ \mathbb E[Z_i\mid X_i=x_i] = \begin{bmatrix} \gamma_{i1}\\ \vdots\\ \gamma_{iK} \end{bmatrix} $$
gives the memberships of one observed point.

In contrast,
$$ \mathbb E[X_i\mid C_i=k]=\mu_k $$
gives the mean position of cluster $k$.

Thus,
$$ \begin{aligned} \mathbb E[Z_i\mid X_i=x_i] &=\text{membership information},\\ \mathbb E[X_i\mid C_i=k] &=\text{location information}. \end{aligned} $$

## 6. Complete-Data Likelihood

If $z_{ik}$ were observed,
$$ p(X,Z) = \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathscr N(x_i\mid\mu_k,\Sigma_k) \right]^{z_{ik}}. $$

Take logarithms:
$$ \ell_c = \log p(X,Z). $$

Using $\log\prod_r a_r=\sum_r\log a_r$,
$$ \ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \log \left[ \phi_k \mathscr N(x_i\mid\mu_k,\Sigma_k) \right]. $$

Using $\log(ab)=\log a+\log b$,
$$ \ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \left[ \log\phi_k+ \log\mathscr N(x_i\mid\mu_k,\Sigma_k) \right]. $$

For each $i$, exactly one $z_{ik}$ equals $1$:
$$ \sum_{k=1}^{K}z_{ik}=1. $$

If the labels were known,
$$ \phi_k = \frac{\text{number assigned to }k}{n}, $$
$$ \mu_k = \text{mean of points assigned to }k, $$
$$ \Sigma_k = \text{covariance of points assigned to }k. $$

Hence,
$$ \text{known labels} \Rightarrow \text{separate Gaussian estimation for each cluster}. $$

## 7. EM Interpretation

Let the current parameter estimates be
$$ \Theta^{(t)} = \left\{ \phi_k^{(t)}, \mu_k^{(t)}, \Sigma_k^{(t)} \right\}_{k=1}^{K}. $$

### E-step
Calculate
$$ \gamma_{ik}^{(t)} = \frac{ \phi_k^{(t)} \mathscr N \left( x_i\mid\mu_k^{(t)},\Sigma_k^{(t)} \right) }{ \displaystyle \sum_{j=1}^{K} \phi_j^{(t)} \mathscr N \left( x_i\mid\mu_j^{(t)},\Sigma_j^{(t)} \right) }. $$

Replace $z_{ik}$ with
$$ \mathbb E[Z_{ik}\mid X_i=x_i,\Theta^{(t)}] = \gamma_{ik}^{(t)}. $$

The expected complete-data log-likelihood is
$$ Q(\Theta\mid\Theta^{(t)}) = \mathbb E_{Z\mid X,\Theta^{(t)}} [\ell_c(\Theta)]. $$

Therefore,
$$ Q = \sum_{i=1}^{n} \sum_{k=1}^{K} \gamma_{ik}^{(t)} \left[ \log\phi_k + \log\mathscr N(x_i\mid\mu_k,\Sigma_k) \right]. $$

The E-step performs
$$ \phi_k^{(t)} \xrightarrow{\text{observe }x_i} \gamma_{ik}^{(t)}. $$

Thus,
$$ \text{prior membership probability} \rightarrow \text{posterior membership probability}. $$

## 8. M-Step Parameter Updates

Define
$$ N_k = \sum_{i=1}^{n}\gamma_{ik}. $$

Because $\sum_{k=1}^{K}\gamma_{ik}=1$,
$$ \sum_{k=1}^{K}N_k = \sum_{i=1}^{n}\sum_{k=1}^{K}\gamma_{ik} = n. $$

### 8.1 Mixture weights
The relevant term is
$$ Q_\phi = \sum_{k=1}^{K} N_k\log\phi_k. $$

Constraint:
$$ \sum_{k=1}^{K}\phi_k=1. $$

Lagrangian:
$$ \mathcal L = \sum_{k=1}^{K} N_k\log\phi_k + \lambda \left( \sum_{k=1}^{K}\phi_k-1 \right). $$

Differentiate:
$$ \frac{\partial\mathcal L}{\partial\phi_k} = \frac{N_k}{\phi_k}+\lambda=0. $$

Thus,
$$ \phi_k=-\frac{N_k}{\lambda}. $$

Using $\sum_{k=1}^{K}\phi_k=1$,
$$ -\frac{1}{\lambda}\sum_{k=1}^{K}N_k=1. $$

Since $\sum_{k=1}^{K}N_k=n$,
$$ \lambda=-n. $$

Therefore,
$$ \phi_k^{\mathrm{new}} = \frac{N_k}{n}. $$

### 8.2 Mean vectors
The mean-dependent term is
$$ Q_{\mu_k} = -\frac12 \sum_{i=1}^{n} \gamma_{ik} (x_i-\mu_k)^T \Sigma_k^{-1} (x_i-\mu_k). $$

Differentiate:
$$ \frac{\partial Q}{\partial\mu_k} = \Sigma_k^{-1} \sum_{i=1}^{n} \gamma_{ik}(x_i-\mu_k). $$

Set equal to zero:
$$ \sum_{i=1}^{n}\gamma_{ik}x_i = \mu_k \sum_{i=1}^{n}\gamma_{ik} $$

Hence,
$$ \mu_k^{\mathrm{new}} = \frac{ \displaystyle\sum_{i=1}^{n}\gamma_{ik}x_i }{ N_k }. $$

### 8.3 Covariance matrices
Maximisation with respect to $\Sigma_k$ gives
$$ \Sigma_k^{\mathrm{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} (x_i-\mu_k^{\mathrm{new}}) (x_i-\mu_k^{\mathrm{new}})^T. $$

The complete M-step is
$$ N_k=\sum_{i=1}^{n}\gamma_{ik} $$
$$ \phi_k^{\mathrm{new}}=\frac{N_k}{n} $$
$$ \mu_k^{\mathrm{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik}x_i $$
$$ \Sigma_k^{\mathrm{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} (x_i-\mu_k^{\mathrm{new}}) (x_i-\mu_k^{\mathrm{new}})^T. $$

If $\gamma_{ik}=0.75$, then $x_i$ contributes $75\%$ of one observation to cluster $k$.
Thus,
$$ \gamma_{ik} = \text{fractional membership weight}. $$

## 9. Interpretation

The mixture weight $\phi_k=P(C_i=k)$ is the prior probability of cluster $k$. The Gaussian density $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ measures compatibility between $x_i$ and cluster $k$. Bayes’ rule produces
$$ \gamma_{ik} = P(C_i=k\mid X_i=x_i), $$
the posterior membership probability. Therefore,
$$ \mathbb E[Z_i\mid X_i=x_i] = (\gamma_{i1},\ldots,\gamma_{iK})^T. $$

The M-step updates $\phi_k,\qquad\mu_k,\qquad\Sigma_k$ using $\gamma_{ik}$ as fractional weights. EM repeats
$$ \text{conditional membership update} \longrightarrow \text{weighted parameter update}. $$

Hence,
$$ \text{GMM clustering} = \text{probabilistic clustering based on } \mathbb E[Z_i\mid X_i=x_i]. $$

The latent labels are updated using Bayes’ rule. The fixed model parameters are estimated by maximum likelihood through EM.

## 10. Computational Simulation

In [10]:
import kagglehub
import pandas as pd
import numpy as np

csv_path = kagglehub.dataset_download(
    "arjunbhasin2013/ccdata",
    path="CC GENERAL.csv"
)

print("Dataset path:", csv_path)

raw_df = pd.read_csv(csv_path)

print("Raw shape:", raw_df.shape)
display(raw_df.head())

Using Colab cache for faster access to the 'ccdata' dataset.
Dataset path: /kaggle/input/ccdata/CC GENERAL.csv
Raw shape: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [11]:
FEATURES = [
    "PURCHASES",
    "CREDIT_LIMIT"
]

selected_df = raw_df[FEATURES].copy()

print(selected_df.info())

print("\nMissing values:")
display(selected_df.isna().sum())

print("\nDescriptive statistics:")
display(selected_df.describe().T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PURCHASES     8950 non-null   float64
 1   CREDIT_LIMIT  8949 non-null   float64
dtypes: float64(2)
memory usage: 140.0 KB
None

Missing values:


,0
PURCHASES,0
CREDIT_LIMIT,1



Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
PURCHASES,8950.0,1003.204834,2136.634782,0.0,39.635,361.28,1110.13,49039.57
CREDIT_LIMIT,8949.0,4494.449450,3638.815725,50.0,1600.000,3000.00,6500.00,30000.00


In [12]:
from datawash import DataPipeline
from pathlib import Path

selected_df = selected_df.replace(
    [np.inf, -np.inf],
    np.nan
)

needs_cleaning = (
    selected_df.isna().any().any()
    or
    any(
        not pd.api.types.is_numeric_dtype(
            selected_df[column]
        )
        for column in FEATURES
    )
)

if needs_cleaning:

    raw_feature_path = (
        "/content/gmm_selected_features_raw.csv"
    )

    cleaned_feature_path = (
        "/content/gmm_selected_features_cleaned.csv"
    )

    selected_df.to_csv(
        raw_feature_path,
        index=False
    )

    pipe = DataPipeline()

    pipe.load_data(
        raw_feature_path
    )

    (
        pipe
        .sanitize_garbage()
        .auto_type_correct()
    )

    pipe.handle_missing_values(
        strategy="auto"
    )

    pipe.save_data(
        cleaned_feature_path
    )

    gmm_df = pd.read_csv(
        cleaned_feature_path
    )

    print("DataWash cleaning applied.")

else:

    gmm_df = selected_df.copy()

    print(
        "No missing or invalid values detected. "
        "Cleaning was not required."
    )

# Final safety conversion
gmm_df[FEATURES] = (
    gmm_df[FEATURES]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

gmm_df = (
    gmm_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna(
        subset=FEATURES
    )
    .reset_index(drop=True)
)

print("Final modelling shape:", gmm_df.shape)
display(gmm_df.head())




Processing "/content/gmm_selected_features_raw.csv"...
INFO:datawash:
Processing "/content/gmm_selected_features_raw.csv"...
Success! Loaded dataset with 8950 rows and 2 columns.
INFO:datawash:Success! Loaded dataset with 8950 rows and 2 columns.
Sanitization complete: Garbage strings and pure whitespace converted to NaNs.
INFO:datawash:Sanitization complete: Garbage strings and pure whitespace converted to NaNs.
No columns required type conversion.
INFO:datawash:No columns required type conversion.

Imputation Strategy: 'auto'
INFO:datawash:
Imputation Strategy: 'auto'
  Successfully imputed: CREDIT_LIMIT
INFO:datawash:  Successfully imputed: CREDIT_LIMIT
Data saved to '/content/gmm_selected_features_cleaned.csv' (csv).
INFO:datawash:Data saved to '/content/gmm_selected_features_cleaned.csv' (csv).


DataWash cleaning applied.
Final modelling shape: (8950, 2)


,PURCHASES,CREDIT_LIMIT
0,95.40,1000.0
1,0.00,7000.0
2,773.17,7500.0
3,1499.00,7500.0
4,16.00,1200.0


In [13]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:

    def __init__(
        self,
        features=(
            "PURCHASES",
            "CREDIT_LIMIT"
        ),
        n_components=3,
        test_size=0.20,
        random_state=42,
        covariance_type="full"
    ):

        self.features = tuple(features)
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state
        self.covariance_type = covariance_type

        self.scaler = StandardScaler()

        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            n_init=10,
            max_iter=500,
            tol=1e-4,
            reg_covar=1e-6,
            random_state=random_state
        )

        self.is_fitted = False

    # =====================================================
    # Data preparation and model fitting
    # =====================================================
    def fit(self, data):

        if isinstance(data, str):

            dataframe = pd.read_csv(data)

        elif isinstance(data, pd.DataFrame):

            dataframe = data.copy()

        else:

            raise TypeError(
                "data must be a DataFrame "
                "or a CSV file path."
            )

        missing_columns = [
            feature
            for feature in self.features
            if feature not in dataframe.columns
        ]

        if missing_columns:

            raise ValueError(
                f"Missing columns: {missing_columns}"
            )

        X = dataframe[
            list(self.features)
        ].copy()

        X = X.apply(
            pd.to_numeric,
            errors="coerce"
        )

        X = (
            X
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .reset_index(drop=True)
        )

        if len(X) < self.n_components:

            raise ValueError(
                "Not enough observations."
            )

        self.data_raw = X

        (
            self.train_raw,
            self.test_raw
        ) = train_test_split(
            self.data_raw,
            test_size=self.test_size,
            random_state=self.random_state,
            shuffle=True
        )

        self.train_raw = (
            self.train_raw
            .reset_index(drop=True)
        )

        self.test_raw = (
            self.test_raw
            .reset_index(drop=True)
        )

        # Fit scaler only on training data
        self.train_scaled = (
            self.scaler.fit_transform(
                self.train_raw.to_numpy()
            )
        )

        self.test_scaled = (
            self.scaler.transform(
                self.test_raw.to_numpy()
            )
        )

        # EM fitting
        self.model.fit(
            self.train_scaled
        )

        # Posterior responsibilities
        self.train_responsibilities = (
            self.model.predict_proba(
                self.train_scaled
            )
        )

        self.test_responsibilities = (
            self.model.predict_proba(
                self.test_scaled
            )
        )

        # Hard labels
        self.train_labels = np.argmax(
            self.train_responsibilities,
            axis=1
        )

        self.test_labels = np.argmax(
            self.test_responsibilities,
            axis=1
        )

        # Maximum responsibilities
        self.train_confidence = np.max(
            self.train_responsibilities,
            axis=1
        )

        self.test_confidence = np.max(
            self.test_responsibilities,
            axis=1
        )

        # Per-sample average log-likelihood
        self.train_log_likelihood = (
            self.model.score(
                self.train_scaled
            )
        )

        self.test_log_likelihood = (
            self.model.score(
                self.test_scaled
            )
        )

        # Means in original feature units
        self.cluster_centres_raw = (
            self.scaler.inverse_transform(
                self.model.means_
            )
        )

        self.is_fitted = True

        print(
            "Converged:",
            self.model.converged_
        )

        print(
            "EM iterations:",
            self.model.n_iter_
        )

        print(
            "Training observations:",
            len(self.train_raw)
        )

        print(
            "Test observations:",
            len(self.test_raw)
        )

        print(
            "Average training log-likelihood:",
            round(
                self.train_log_likelihood,
                5
            )
        )

        print(
            "Average test log-likelihood:",
            round(
                self.test_log_likelihood,
                5
            )
        )

        return self

    # =====================================================
    # Fitted-state validation
    # =====================================================
    def _check_fitted(self):

        if not self.is_fitted:

            raise RuntimeError(
                "Call fit() first."
            )

    # =====================================================
    # Empirical 2D density
    # =====================================================
    def plot_empirical_density(self):

        self._check_fitted()

        x_feature, y_feature = self.features

        fig = px.density_heatmap(
            self.train_raw,
            x=x_feature,
            y=y_feature,
            nbinsx=60,
            nbinsy=60,
            marginal_x="histogram",
            marginal_y="histogram",
            title=(
                "Empirical Density of "
                "Training Customers"
            )
        )

        fig.update_layout(
            template="plotly_white",
            width=950,
            height=700
        )

        return fig

    # =====================================================
    # Construct grid and responsibilities
    # =====================================================
    def _create_grid(
        self,
        grid_size=250,
        quantile_window=(0.005, 0.995)
    ):

        self._check_fitted()

        x_feature, y_feature = self.features

        if quantile_window is None:

            x_min = self.data_raw[
                x_feature
            ].min()

            x_max = self.data_raw[
                x_feature
            ].max()

            y_min = self.data_raw[
                y_feature
            ].min()

            y_max = self.data_raw[
                y_feature
            ].max()

        else:

            lower, upper = quantile_window

            x_min, x_max = (
                self.data_raw[x_feature]
                .quantile(
                    [lower, upper]
                )
            )

            y_min, y_max = (
                self.data_raw[y_feature]
                .quantile(
                    [lower, upper]
                )
            )

        x_range = max(
            x_max - x_min,
            1.0
        )

        y_range = max(
            y_max - y_min,
            1.0
        )

        x_padding = 0.05 * x_range
        y_padding = 0.05 * y_range

        x_min -= x_padding
        x_max += x_padding

        y_min -= y_padding
        y_max += y_padding

        x_values = np.linspace(
            x_min,
            x_max,
            grid_size
        )

        y_values = np.linspace(
            y_min,
            y_max,
            grid_size
        )

        xx, yy = np.meshgrid(
            x_values,
            y_values
        )

        grid_raw = np.column_stack(
            [
                xx.ravel(),
                yy.ravel()
            ]
        )

        grid_scaled = (
            self.scaler.transform(
                grid_raw
            )
        )

        grid_responsibilities = (
            self.model.predict_proba(
                grid_scaled
            )
        )

        maximum_responsibility = np.max(
            grid_responsibilities,
            axis=1
        ).reshape(xx.shape)

        hard_cluster = np.argmax(
            grid_responsibilities,
            axis=1
        ).reshape(xx.shape)

        return {
            "x_values": x_values,
            "y_values": y_values,
            "maximum_responsibility":
                maximum_responsibility,
            "hard_cluster": hard_cluster,
            "bounds": (
                x_min,
                x_max,
                y_min,
                y_max
            )
        }

    # =====================================================
    # General assignment visualisation
    # =====================================================
    def _plot_assignments(
        self,
        raw_data,
        labels,
        responsibilities,
        title,
        grid_size=250
    ):

        surface = self._create_grid(
            grid_size=grid_size
        )

        x_feature, y_feature = self.features

        (
            x_min,
            x_max,
            y_min,
            y_max
        ) = surface["bounds"]

        display_mask = (
            raw_data[x_feature].between(
                x_min,
                x_max
            )
            &
            raw_data[y_feature].between(
                y_min,
                y_max
            )
        ).to_numpy()

        shown_data = (
            raw_data
            .loc[display_mask]
            .reset_index(drop=True)
        )

        shown_labels = labels[
            display_mask
        ]

        shown_responsibilities = (
            responsibilities[
                display_mask
            ]
        )

        shown_confidence = np.max(
            shown_responsibilities,
            axis=1
        )

        fig = go.Figure()

        # Continuous soft-assignment confidence
        fig.add_trace(
            go.Contour(
                x=surface["x_values"],
                y=surface["y_values"],
                z=surface[
                    "maximum_responsibility"
                ],
                colorscale="Viridis",
                opacity=0.72,
                contours=dict(
                    start=1.0
                    / self.n_components,
                    end=1.0,
                    size=0.05,
                    coloring="heatmap",
                    showlabels=False
                ),
                colorbar=dict(
                    title=(
                        "max posterior"
                        "<br>responsibility"
                    )
                ),
                name="Soft confidence"
            )
        )

        # Hard-cluster decision boundaries
        if self.n_components > 1:

            fig.add_trace(
                go.Contour(
                    x=surface["x_values"],
                    y=surface["y_values"],
                    z=surface["hard_cluster"],
                    contours=dict(
                        start=0.5,
                        end=(
                            self.n_components
                            - 1.5
                        ),
                        size=1.0,
                        coloring="none"
                    ),
                    line=dict(
                        color="black",
                        width=1.5
                    ),
                    showscale=False,
                    hoverinfo="skip",
                    name="Hard boundaries"
                )
            )

        # Points grouped by hard cluster
        for cluster_id in range(
            self.n_components
        ):

            cluster_mask = (
                shown_labels
                == cluster_id
            )

            cluster_points = (
                shown_data
                .loc[cluster_mask]
            )

            cluster_confidence = (
                shown_confidence[
                    cluster_mask
                ]
            )

            fig.add_trace(
                go.Scattergl(
                    x=cluster_points[
                        x_feature
                    ],
                    y=cluster_points[
                        y_feature
                    ],
                    mode="markers",
                    marker=dict(
                        size=5,
                        opacity=0.70
                    ),
                    customdata=(
                        cluster_confidence
                    ),
                    hovertemplate=(
                        f"{x_feature}: "
                        "%{x:.2f}<br>"
                        f"{y_feature}: "
                        "%{y:.2f}<br>"
                        f"Cluster: "
                        f"{cluster_id}<br>"
                        "Maximum responsibility: "
                        "%{customdata:.3f}"
                        "<extra></extra>"
                    ),
                    name=(
                        f"Cluster "
                        f"{cluster_id}"
                    )
                )
            )

        # Component means
        fig.add_trace(
            go.Scatter(
                x=self.cluster_centres_raw[
                    :, 0
                ],
                y=self.cluster_centres_raw[
                    :, 1
                ],
                mode="markers+text",
                marker=dict(
                    symbol="x",
                    size=17,
                    color="black",
                    line=dict(
                        width=2
                    )
                ),
                text=[
                    rf"μ{k}"
                    for k in range(
                        self.n_components
                    )
                ],
                textposition="top center",
                name="Component means"
            )
        )

        fig.update_layout(
            title=title,
            xaxis_title=x_feature,
            yaxis_title=y_feature,
            template="plotly_white",
            width=1000,
            height=650
        )

        fig.update_xaxes(
            range=[
                x_min,
                x_max
            ]
        )

        fig.update_yaxes(
            range=[
                y_min,
                y_max
            ]
        )

        return fig

    # =====================================================
    # Training assignment plot
    # =====================================================
    def plot_training_assignments(
        self,
        grid_size=250
    ):

        self._check_fitted()

        return self._plot_assignments(
            raw_data=self.train_raw,
            labels=self.train_labels,
            responsibilities=(
                self.train_responsibilities
            ),
            title=(
                "Training Data: "
                "GMM Soft Responsibilities"
            ),
            grid_size=grid_size
        )

    # =====================================================
    # Test assignment plot
    # =====================================================
    def plot_test_assignments(
        self,
        grid_size=250
    ):

        self._check_fitted()

        return self._plot_assignments(
            raw_data=self.test_raw,
            labels=self.test_labels,
            responsibilities=(
                self.test_responsibilities
            ),
            title=(
                "Unseen Test Data: "
                "GMM Soft Responsibilities"
            ),
            grid_size=grid_size
        )

    # =====================================================
    # Cluster summary
    # =====================================================
    def cluster_summary(self):

        self._check_fitted()

        summary_data = (
            self.train_raw.copy()
        )

        summary_data["Cluster"] = (
            self.train_labels
        )

        summary_data[
            "Maximum responsibility"
        ] = self.train_confidence

        summary = (
            summary_data
            .groupby("Cluster")
            .agg(
                Count=(
                    self.features[0],
                    "size"
                ),
                Mean_PURCHASES=(
                    self.features[0],
                    "mean"
                ),
                Mean_CREDIT_LIMIT=(
                    self.features[1],
                    "mean"
                ),
                Mean_Confidence=(
                    "Maximum responsibility",
                    "mean"
                )
            )
        )

        return summary

In [14]:
segmenter = GMMFinancialSegmenter(
    features=(
        "PURCHASES",
        "CREDIT_LIMIT"
    ),
    n_components=3,
    test_size=0.20,
    random_state=42,
    covariance_type="full"
)

segmenter.fit(
    gmm_df
)

Converged: True
EM iterations: 31
Training observations: 7160
Test observations: 1790
Average training log-likelihood: -1.58979
Average test log-likelihood: -1.59019


In [15]:
print(
    "Converged:",
    segmenter.model.converged_
)

print(
    "Iterations:",
    segmenter.model.n_iter_
)

print(
    "Training average log-likelihood:",
    segmenter.train_log_likelihood
)

print(
    "Test average log-likelihood:",
    segmenter.test_log_likelihood
)

print(
    "Train-test score difference:",
    (
        segmenter.train_log_likelihood
        -
        segmenter.test_log_likelihood
    )
)

# Cluster summary:
display(
    segmenter
    .cluster_summary()
    .round(3)
)

Converged: True
Iterations: 31
Training average log-likelihood: -1.5897852131476478
Test average log-likelihood: -1.5901870296980805
Train-test score difference: 0.0004018165504326987


,Count,Mean_PURCHASES,Mean_CREDIT_LIMIT,Mean_Confidence
Cluster,,,,
0,3194,177.173,2043.627,0.916
1,689,5176.147,9816.981,0.923
2,3277,907.336,5842.654,0.914


In [18]:
fig_density = (
    segmenter
    .plot_empirical_density()
)

fig_density.show()

In [19]:
fig_training = (
    segmenter
    .plot_training_assignments(
        grid_size=250
    )
)

fig_training.show()

At each grid point, $x_{\text{grid}}$, the model computes
$$ \mathbb E[Z\mid X=x_{\text{grid}}] = \begin{bmatrix} \gamma_1(x_{\text{grid}})\\ \gamma_2(x_{\text{grid}})\\ \gamma_3(x_{\text{grid}}) \end{bmatrix}. $$

The background displays
$$ c(x_{\text{grid}}) = \max_{1\leq k\leq3} \gamma_k(x_{\text{grid}}). $$

Since $\sum_{k=1}^{3}\gamma_k=1$,
$$ \frac13 \leq c(x_{\text{grid}}) \leq1. $$

Therefore,
$c(x_{\text{grid}})\approx1$ means one component dominates.
$c(x_{\text{grid}})\approx\frac13$ means strong ambiguity.

The black boundaries show
$$ \operatorname*{arg\,max}_{k} \gamma_k(x_{\text{grid}}) $$
changing from one component to another.


In [20]:
fig_test = (
    segmenter
    .plot_test_assignments(
        grid_size=250
    )
)

fig_test.show()

The GMM parameters are estimated using only $X_{\text{train}}$. For each unseen point, $x_i^{\text{test}}$, the model computes
$$ \gamma_{ik}^{\text{test}} = P( C_i=k \mid X_i=x_i^{\text{test}} ). $$

Thus,
$$ \text{test plot} = \text{out-of-sample conditional membership update}. $$

Test points near boundaries satisfy
$$ \gamma_{i1} \approx \gamma_{i2}, $$
or a similar equality between components. These are uncertain assignments.

